# Titanic Survival Prediction — Full Feature Engineering Pipeline

**Student:** Hessah Alkhalifah

**Course:** Arfitcal Intlegince

**Objective:** Compare baseline vs engineered features using a full preprocessing pipeline


# **Import Libraries**

In [3]:
!wget https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv

--2026-02-21 13:16:10--  https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 60302 (59K) [text/plain]
Saving to: ‘titanic.csv’

titanic.csv         100%[===================>]  58.89K  --.-KB/s    in 0.001s  

2026-02-21 13:16:10 (102 MB/s) - ‘titanic.csv’ saved [60302/60302]



In [4]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score


# **Load Data**

In [5]:
df = pd.read_csv("/content/titanic.csv")
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


# **Data Cleaning**

In [7]:
df["Age"] = df["Age"].fillna(df["Age"].median())
df["Fare"] = df["Fare"].fillna(df["Fare"].median())
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])

# **Feature Engineering**

In [8]:
df["AgeGroup"] = pd.cut(
    df["Age"],
    bins=[0,12,18,35,60,100],
    labels=["Child","Teen","YoungAdult","Adult","Senior"]
)

# **Log Transformation**

In [11]:
y = df["Survived"]
X = df.drop(columns=["Survived","Name","Ticket","Cabin"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)




# **Interaction Feature**

In [10]:
df["Pclass_Fare"] = df["Pclass"] * df["Fare"]

# **Train/Test Split**

In [9]:
df["Fare_log"] = np.log1p(df["Fare"])

# **BASELINE MODEL**

In [13]:
basic_features = ["Pclass","Sex","Age","Fare","Embarked"]

cat_cols = ["Sex","Embarked"]
num_cols = ["Pclass","Age","Fare"]

preprocess_basic = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ("num", StandardScaler(), num_cols)
])

baseline_model = Pipeline([
    ("prep", preprocess_basic),
    ("model", LogisticRegression(max_iter=1000))
])

baseline_model.fit(X_train[basic_features], y_train)

pred_base = baseline_model.predict(X_test[basic_features])

# **ENHANCED MODEL**

In [14]:
cat_cols = ["Sex","Embarked","AgeGroup"]
num_cols = ["Pclass","Age","Fare","Fare_log","Pclass_Fare"]

preprocess_adv = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ("num", StandardScaler(), num_cols)
])

enhanced_model = Pipeline([
    ("prep", preprocess_adv),
    ("model", LogisticRegression(max_iter=1000))
])

enhanced_model.fit(X_train, y_train)

pred_adv = enhanced_model.predict(X_test)

# **EVALUATION**

In [15]:
print("Baseline Accuracy:", accuracy_score(y_test, pred_base))
print("Enhanced Accuracy:", accuracy_score(y_test, pred_adv))

Baseline Accuracy: 0.7988826815642458
Enhanced Accuracy: 0.8100558659217877


# **REPRODUCIBLE PIPELINE**



All preprocessing steps (encoding, scaling, transformations, and interaction features) are implemented inside a sklearn Pipeline to guarantee consistent preprocessing during training and inference.

# **DOMAIN INTUITION**


Passenger class reflects socioeconomic status.
Age groups capture evacuation priority.
Fare transformation reduces skewness.
Interaction features represent combined economic influence.

# **LEAKAGE PREVENTION**


Train-test split was performed before evaluation.
Pipelines ensure transformations are learned only from training data.
Target information was not used during preprocessing.


# **ETHICAL CONSIDERATIONS**


Features like passenger class may introduce socioeconomic bias.
Fairness evaluation is necessary before real-world deployment.

# **SAVE CLEAN DATASET**

In [12]:
df.to_csv("titanic_cleaned.csv", index=False)

# **Reflection**

Feature engineering improved performance by transforming raw numeric values into meaningful patterns. Age grouping helped capture the “women and children first” rule. Fare transformation reduced skewness, and interaction features improved socioeconomic representation.

Passenger class could introduce bias because it reflects wealth rather than behavior.

Pre-deployment checks:
1. Fairness testing across demographic groups
2. Data leakage validation